# k2 — Dịch 1.200 câu hỏi vi → en → zh, so sánh model / phiên bản prompt (GPU T4 ×2)

**Cài đặt:** Accelerator = **GPU T4 ×2**, Internet = On. **Add Input:** `mir2-code`, `mir2-data`
(+ `mir2-qtrans` bản trước: để so với bản cũ, để phiếu chấm mới không lặp câu đã chấm, và để chạy tiếp khi bị ngắt).

Mỗi model chạy 2 tiến trình song song (mỗi GPU một bản 4-bit, mỗi bản một nửa số câu). Kết quả ở
`qtrans/<model>-<prompt>/` (prompt `t1` cũ giữ tên `qtrans/<model>/`).
**Prompt t2** (mặc định): sửa theo lỗi đã gặp khi so Qwen2.5-14B / Qwen3-8B với t1 — quy tắc thuật ngữ, triệu chứng không
nâng thành chẩn đoán, đủ ý / không thêm, đúng người, chữ số, chữ viết tắt, biệt dược; gợi ý thuật ngữ đã kiểm chứng theo
từng câu (`configs/med_glossary_vi.tsv`). Đo thực tế với t1: 14B ~2 giờ, 8B ~1,2 giờ cho 1.200 câu (lô 8).
Ô (1): `LIMIT = 20` chạy thử để đọc mẫu, rồi `LIMIT = 0`. `COMPARE` = các cặp cần so (tên thư mục trong qtrans/).

**Đầu ra:** `qtrans/<khoá>/records-*.jsonl`, `qtrans/compare_*/{auto_metrics.md, sheet.csv, key.json}` → dataset
**`mir2-qtrans`** (bản mới). Phiếu `sheet.csv` chấm mù (không mở `key.json`), rồi
`python -m mirret.translate score --sheet <phiếu đã chấm>` (docs/EVAL_PLAN.md mục 0).

In [ ]:
# (1) Chuẩn bị. Chỉ sửa các biến viết HOA ngay dưới đây.
NB_VERSION = 'k2-v3 (15/10/2026)'
MODELS = ['qwen25_14b', 'qwen3_14b']  # tên trong translate.models hoặc id Hugging Face
PROMPT = 't2'
LIMIT = 0
BATCH = {'qwen25_14b': 8, 'qwen3_14b': 8, 'qwen3_8b': 16}   # câu/lô mỗi GPU; hết bộ nhớ → tự giảm
QUANT = '4bit'                       # '4bit' (nhanh nhất trên T4) | '8bit' (chậm hơn)
COMPARE = [('qwen25_14b-t2', 'qwen3_14b-t2'),   # so 2 model cùng prompt
           ('qwen25_14b', 'qwen25_14b-t2')]     # so prompt t1 → t2 (cần mir2-qtrans bản cũ)
import glob, os, shutil, sys, zipfile
KAGGLE = os.environ.get('KAGGLE_ROOT', '/kaggle')            # chạy thử ngoài Kaggle: đặt biến môi trường KAGGLE_ROOT
CODE = f'{KAGGLE}/working/code'
shutil.rmtree(CODE, ignore_errors=True)
hits = sorted(glob.glob(f'{KAGGLE}/input/**/mirret/__init__.py', recursive=True), key=len)
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE)
else:
    zips = sorted(glob.glob(f'{KAGGLE}/input/**/mir2_code.zip', recursive=True))
    assert zips, 'Chưa thêm dataset mir2-code (Add Input → Your Datasets)'
    zipfile.ZipFile(zips[0]).extractall(CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
from mirret.kaggle_nb import setup, sh, par, locate, locate_all, report
K = setup(NB_VERSION, {'translate.prompt': PROMPT}, inputs=())

In [ ]:
# (2) Chép bản ghi + phiếu so sánh của các phiên trước (nếu có); cài bitsandbytes (4-bit)
for d in locate_all('qtrans'):
    for f in glob.glob(f'{d}/qtrans/*/records*.jsonl') + glob.glob(f'{d}/qtrans/compare_*/*'):
        dst = f'{K.work}/qtrans/' + os.path.relpath(f, f'{d}/qtrans')
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(f, dst); print('chép', f)
sh('python -m pip install -q -U bitsandbytes accelerate')

In [ ]:
# (3) Sinh bản dịch cho từng model, 2 GPU song song
for m in MODELS:
    lim = f' --limit {LIMIT}' if LIMIT else ''
    bs = f' --batch {BATCH[m]}' if m in BATCH else ''
    par([f'CUDA_VISIBLE_DEVICES={g} python -m mirret.translate generate --config {K.cfg} --model {m} --shard {g}/2'
         f'{lim}{bs} --quant {QUANT} --prompt {PROMPT}' for g in range(2)], f'translate_{m}')

In [ ]:
# (4) So sánh: chỉ số tự động + phiếu chấm mù MỚI (bỏ các câu đã chấm ở lần trước)
for a, b in COMPARE:
    if all(os.path.isdir(f'{K.work}/qtrans/{x}') for x in (a, b)):
        sh(f'python -m mirret.translate compare --config {K.cfg} --a {a} --b {b} --n 100')
    else:
        print('bỏ qua so sánh', a, b, '(thiếu thư mục bản dịch)')
report('k2')